In [1]:

import pandas as pd
import numpy as np
from datetime import timedelta

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

print("Libraries imported successfully!")


Libraries imported successfully!


In [2]:

df = pd.read_csv("demandsense_cleaned.csv")

df["date"] = pd.to_datetime(df["date"])

print("Dataset loaded successfully!")
print("Total rows:", len(df))

print("\nAvailable columns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


Dataset loaded successfully!
Total rows: 6905

Available columns:
['date', 'hotel_id', 'hotel_name', 'total_rooms', 'available_rooms', 'occupied_rooms', 'occupancy_rate', 'bookings', 'bookings_cancelled', 'avg_price', 'day_of_week', 'month', 'is_weekend', 'is_holiday', 'festival_or_event', 'weather', 'tourist_arrivals', 'data_type', 'year', 'day', 'day_of_year']

First 5 rows:


,date,hotel_id,hotel_name,total_rooms,available_rooms,occupied_rooms,occupancy_rate,bookings,bookings_cancelled,avg_price,...,month,is_weekend,is_holiday,festival_or_event,weather,tourist_arrivals,data_type,year,day,day_of_year
0,2023-01-01,H001,Mountain View Hotel,80,80,54,67.50,36,7,4747.33,...,1,1,1,New Year,Rainy,794,synthetic_demo,2023,1,1
1,2023-01-02,H001,Mountain View Hotel,80,80,38,47.50,41,5,4701.70,...,1,0,1,New Year,Rainy,701,synthetic_demo,2023,2,2
2,2023-01-03,H001,Mountain View Hotel,80,80,37,46.25,41,3,4172.83,...,1,0,0,No event,Clear,409,synthetic_demo,2023,3,3
3,2023-01-04,H001,Mountain View Hotel,80,80,36,45.00,34,2,4253.72,...,1,0,0,No event,Clear,586,synthetic_demo,2023,4,4
4,2023-01-05,H001,Mountain View Hotel,80,78,45,57.69,38,2,4290.55,...,1,0,0,No event,Clear,519,synthetic_demo,2023,5,5


In [3]:

print("Available hotel IDs:")
print(df["hotel_id"].unique())

print("\nEarliest date:", df["date"].min().date())
print("Latest date:", df["date"].max().date())

print("\nMissing occupancy values:")
print(df["occupancy_rate"].isnull().sum())


Available hotel IDs:
['H001' 'H002' 'H003' 'H004' 'H005']

Earliest date: 2023-01-01
Latest date: 2026-10-12

Missing occupancy values:
0


In [4]:

# Select one hotel for training
hotel_id = "H001"

hotel = df[df["hotel_id"] == hotel_id].copy()

# Sort dates in chronological order
hotel = hotel.sort_values("date").reset_index(drop=True)

# Create date-based features
hotel["month"] = hotel["date"].dt.month
hotel["day"] = hotel["date"].dt.day
hotel["day_of_year"] = hotel["date"].dt.dayofyear

hotel["is_weekend"] = (
    hotel["date"].dt.dayofweek >= 5
).astype(int)

# Create previous-day and previous-week occupancy features
hotel["occupancy_lag_1"] = hotel["occupancy_rate"].shift(1)
hotel["occupancy_lag_7"] = hotel["occupancy_rate"].shift(7)

# Remove rows where lag values are unavailable
hotel = hotel.dropna().reset_index(drop=True)

features = [
    "month",
    "day",
    "day_of_year",
    "is_weekend",
    "is_holiday",
    "occupancy_lag_1",
    "occupancy_lag_7"
]

X = hotel[features]
y = hotel["occupancy_rate"]

print("Hotel selected:", hotel_id)
print("Training rows available:", len(hotel))
print("Features:", features)


Hotel selected: H001
Training rows available: 1374
Features: ['month', 'day', 'day_of_year', 'is_weekend', 'is_holiday', 'occupancy_lag_1', 'occupancy_lag_7']


In [5]:

# Use the first 80% of dates for training
split = int(len(hotel) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# Train the model
model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

# Predict occupancy on the held-out historical dates
predictions = model.predict(X_test)

# Calculate errors
mae = mean_absolute_error(y_test, predictions)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Test MAE:", round(mae, 2), "percentage points")


Training rows: 1099
Testing rows: 275
Test MAE: 6.19 percentage points


In [6]:

final_model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

final_model.fit(X, y)

last_date = hotel["date"].max()

print("Final model trained successfully!")
print("Hotel:", hotel_id)
print("Last historical date:", last_date.date())


Final model trained successfully!
Hotel: H001
Last historical date: 2026-10-12


In [7]:

future_date_input = input(
    "Enter future date (YYYY-MM-DD): "
)

future_date = pd.to_datetime(
    future_date_input,
    errors="raise"
)

if future_date <= last_date:
    print(
        "Please enter a date after",
        last_date.date()
    )
else:
    print("Requested future date:", future_date.date())
    print("Days ahead:", (future_date - last_date).days)



Enter future date (YYYY-MM-DD):  2026-11-20


Requested future date: 2026-11-20
Days ahead: 39


In [13]:

# Store historical occupancy values by date
occupancy_history = dict(
    zip(df[df["hotel_id"] == hotel_id]["date"],
        df[df["hotel_id"] == hotel_id]["occupancy_rate"])
)

# Forecast each day until the requested date
current_date = last_date + timedelta(days=1)
forecast_results = []

while current_date <= future_date:

    previous_day = current_date - timedelta(days=1)
    previous_week = current_date - timedelta(days=7)

    lag_1 = occupancy_history.get(previous_day)
    lag_7 = occupancy_history.get(previous_week)

    if lag_1 is None or lag_7 is None:
        raise ValueError(
            "Missing historical occupancy for lag features."
        )

    # Use known holiday flag if available; otherwise assume 0
    holiday_rows = df[df["date"] == current_date]
    if not holiday_rows.empty and "is_holiday" in holiday_rows.columns:
        is_holiday = int(holiday_rows.iloc[0]["is_holiday"])
    else:
        is_holiday = 0

    future_row = pd.DataFrame([{
        "month": current_date.month,
        "day": current_date.day,
        "day_of_year": current_date.dayofyear,
        "is_weekend": int(current_date.dayofweek >= 5),
        "is_holiday": is_holiday,
        "occupancy_lag_1": lag_1,
        "occupancy_lag_7": lag_7
    }])

    predicted_occupancy = float(
        final_model.predict(future_row[features])[0]
    )

    predicted_occupancy = np.clip(
        predicted_occupancy, 0, 100
    )

    # Save the prediction for use on later forecast days
    occupancy_history[current_date] = predicted_occupancy

    forecast_results.append({
        "date": current_date.date(),
        "predicted_occupancy": round(
            predicted_occupancy, 2
        )
    })

    current_date += timedelta(days=1)

forecast_df = pd.DataFrame(forecast_results)

print("Forecast for", hotel_id)
display(forecast_df.tail(10))

print(
    "\nPredicted occupancy on",
    future_date.date(),
    ":",
    forecast_df.iloc[-1]["predicted_occupancy"],
    "%"
)


Forecast for H001


,date,predicted_occupancy
29,2026-11-11,51.25
30,2026-11-12,51.41
31,2026-11-13,50.81
32,2026-11-14,70.68
33,2026-11-15,70.99
34,2026-11-16,50.90
35,2026-11-17,48.97
36,2026-11-18,50.62
37,2026-11-19,49.12
38,2026-11-20,50.88



Predicted occupancy on 2026-11-20 : 50.88 %


In [14]:

# Convert both dates to the same format
forecast_df["date"] = pd.to_datetime(forecast_df["date"]).dt.date
future_date_only = pd.Timestamp(future_date).date()

# Find the requested date
result = forecast_df[
    forecast_df["date"] == future_date_only
]

if not result.empty:

    occupancy = result.iloc[0]["predicted_occupancy"]

    total_rooms = int(
        df[df["hotel_id"] == hotel_id]["total_rooms"].iloc[0]
    )

    occupied_rooms = round(total_rooms * occupancy / 100)
    empty_rooms = total_rooms - occupied_rooms

    print("=" * 35)
    print("       DEMANDSENSE FORECAST")
    print("=" * 35)
    print("Hotel ID:", hotel_id)
    print("Forecast Date:", future_date_only)
    print("Predicted Occupancy:", occupancy, "%")
    print("Total Rooms:", total_rooms)
    print("Expected Occupied Rooms:", occupied_rooms)
    print("Expected Empty Rooms:", empty_rooms)
    print("=" * 35)

else:
    print("Forecast not found.")
    print("Requested date:", future_date_only)
    print("Last forecast date:", forecast_df["date"].max())


       DEMANDSENSE FORECAST
Hotel ID: H001
Forecast Date: 2026-11-20
Predicted Occupancy: 50.88 %
Total Rooms: 80
Expected Occupied Rooms: 41
Expected Empty Rooms: 39


In [15]:

# Get occupancy forecast for the requested date
result = forecast_df[
    forecast_df["date"] == future_date.date()
]

if result.empty:
    print("Run the forecast cells first.")
else:
    occupancy = float(
        result.iloc[0]["predicted_occupancy"]
    )

    total_rooms = int(
        df[df["hotel_id"] == hotel_id]["total_rooms"].iloc[0]
    )

    expected_occupied = round(total_rooms * occupancy / 100)
    expected_empty = total_rooms - expected_occupied

    print("Forecast date:", future_date.date())
    print("Predicted occupancy:", occupancy, "%")
    print("Expected occupied rooms:", expected_occupied)
    print("Expected empty rooms:", expected_empty)


Forecast date: 2026-11-20
Predicted occupancy: 50.88 %
Expected occupied rooms: 41
Expected empty rooms: 39


In [16]:

# Demo assumptions — not real hotel financial data
room_price = 4000
room_service_cost = 700
direct_acquisition_cost = 200

# Contribution if one additional room is booked directly
contribution_per_booking = (
    room_price
    - room_service_cost
    - direct_acquisition_cost
)

# Maximum theoretical contribution if every empty room is filled
maximum_contribution_opportunity = (
    expected_empty * contribution_per_booking
)

print("Contribution per additional direct booking: ₹",
      contribution_per_booking)

print("Maximum theoretical contribution opportunity: ₹",
      maximum_contribution_opportunity)


Contribution per additional direct booking: ₹ 3100
Maximum theoretical contribution opportunity: ₹ 120900


In [17]:

# Save the forecast so Notebook 04 can read it
forecast_df.to_csv("demandsense_forecast.csv", index=False)

print("Forecast saved successfully!")


Forecast saved successfully!


In [18]:
import os
from pathlib import Path

print("Notebook folder:", os.getcwd())

file_path = Path("demandsense_30_day_forecast.csv")

print("File exists:", file_path.exists())

if file_path.exists():
    print("File size:", file_path.stat().st_size, "bytes")
    print("First 500 characters:")
    print(file_path.read_text()[:500])

Notebook folder: /Users/shivanshkhanna/DemandSense 
File exists: True
File size: 862 bytes
First 500 characters:
date,predicted_occupancy,total_rooms,predicted_occupied_rooms,predicted_vacant_rooms
2026-10-13,50.39,80,40,40
2026-10-14,50.18,80,40,40
2026-10-15,50.18,80,40,40
2026-10-16,50.18,80,40,40
2026-10-17,70.3,80,56,24
2026-10-18,71.73,80,57,23
2026-10-19,50.56,80,40,40
2026-10-20,50.78,80,41,39
2026-10-21,50.63,80,41,39
2026-10-22,50.71,80,41,39
2026-10-23,50.06,80,40,40
2026-10-24,71.08,80,57,23
2026-10-25,72.55,80,58,22
2026-10-26,52.53,80,42,38
2026-10-27,52.5,80,42,38
2026-10-28,53.45,80,43,37
2


In [19]:
import pandas as pd

forecast_df = pd.read_csv("demandsense_30_day_forecast.csv")

print(forecast_df.columns.tolist())
print(forecast_df.head())

['date', 'predicted_occupancy', 'total_rooms', 'predicted_occupied_rooms', 'predicted_vacant_rooms']
         date  predicted_occupancy  total_rooms  predicted_occupied_rooms  \
0  2026-10-13                50.39           80                        40   
1  2026-10-14                50.18           80                        40   
2  2026-10-15                50.18           80                        40   
3  2026-10-16                50.18           80                        40   
4  2026-10-17                70.30           80                        56   

   predicted_vacant_rooms  
0                      40  
1                      40  
2                      40  
3                      40  
4                      24  


In [ ]:
!streamlit run app.py

2026-10-14 02:04:29.655 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.20.10.13:8502

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
